# Graph-Based Feature Engineering

This experiment adds sparse, interpretable relationship history to the current best frequency-plus-historical-amount pipeline. It uses no graph library, target statistics, model tuning, or reserved final-split data.

In [1]:
import gc
import sys
import time
import tracemalloc
from pathlib import Path

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
)
from fraud_detection.features import (
    BehavioralFeatureEngineer,
    GraphFeatureEngineer,
    Phase1FeatureEngineer,
)
from fraud_detection.models import TrainingOnlyPreprocessor, evaluate_probabilities

plt.style.use("seaborn-v0_8-whitegrid")
notebook_started = time.perf_counter()

## Past-only graph semantics

The sparse state tracks `card1↔DeviceInfo` and `card1↔addr1`. For every timestamp, all rows are scored against state from strictly earlier timestamps; only then is the entire timestamp batch committed. Missing endpoints create neither nodes nor edges. TRAIN initializes state and VALIDATION advances it chronologically without labels.

In [2]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validation = split.validation
del transaction, identity, merged
gc.collect()

predictor_columns = [
    column for column in train.columns if column not in ("isFraud", "TransactionID")
]
train_predictors = train.loc[:, predictor_columns]
validation_predictors = validation.loc[:, predictor_columns]
y_train = train["isFraud"].astype("int8")
y_validation = validation["isFraud"].astype("int8")

print(f"Train rows: {len(train):,}")
print(f"Validation rows: {len(validation):,}")
print("Reserved final split status: not assigned or accessed")

Train rows: 413,378
Validation rows: 88,581
Reserved final split status: not assigned or accessed


In [3]:
frequency_engineer = Phase1FeatureEngineer(
    include_amount_time=False,
    include_missing_indicators=False,
    include_frequency_encodings=True,
).fit(train_predictors)
frequency_train = frequency_engineer.transform(train_predictors)
frequency_validation = frequency_engineer.transform(validation_predictors)

behavior_source_columns = [
    column
    for column in (
        "TransactionID",
        "TransactionDT",
        "TransactionAmt",
        "card1",
        "addr1",
        "DeviceInfo",
        "P_emaildomain",
    )
    if column in train.columns
]
amount_engineer = BehavioralFeatureEngineer(
    include_counts=False,
    include_amounts=True,
    include_recency=False,
    include_velocity=False,
)
amount_train = amount_engineer.fit_transform_features(train.loc[:, behavior_source_columns])
amount_validation = amount_engineer.transform_features(
    validation.loc[:, behavior_source_columns], update_state=True
)
base_train = pd.concat([frequency_train, amount_train], axis=1)
base_validation = pd.concat([frequency_validation, amount_validation], axis=1)
print(f"Current-best feature count: {base_train.shape[1]}")

Current-best feature count: 443


In [4]:
graph_source_columns = ["TransactionID", "TransactionDT", "card1", "DeviceInfo", "addr1"]
graph_engineer = GraphFeatureEngineer()
tracemalloc.start()
graph_started = time.perf_counter()
graph_train = graph_engineer.fit_transform_features(train.loc[:, graph_source_columns])
graph_validation = graph_engineer.transform_features(validation.loc[:, graph_source_columns])
graph_generation_seconds = time.perf_counter() - graph_started
_, graph_peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
graph_feature_memory_mb = (
    graph_train.memory_usage(deep=True).sum() + graph_validation.memory_usage(deep=True).sum()
) / 1024**2
graph_peak_memory_mb = graph_peak_bytes / 1024**2

print(f"Graph features generated: {graph_train.shape[1]}")
print(f"Graph generation seconds: {graph_generation_seconds:.1f}")
print(f"Graph feature matrices: {graph_feature_memory_mb:.1f} MiB")
print(f"Traced graph-generation peak: {graph_peak_memory_mb:.1f} MiB")
display(pd.Series(graph_engineer.feature_groups_, name="features"))
display(pd.Series(graph_engineer.relationship_counts, name="state cardinality"))

graph_feature_groups = dict(graph_engineer.feature_groups_)
all_graph_feature_names = graph_engineer.generated_feature_names_
graph_state_counts = graph_engineer.relationship_counts
base_preprocessor = TrainingOnlyPreprocessor()
base_x_train = base_preprocessor.fit_transform(base_train)
base_x_validation = base_preprocessor.transform(base_validation)
graph_x_train = graph_train.to_numpy(dtype="float32", copy=True)
graph_x_validation = graph_validation.to_numpy(dtype="float32", copy=True)

del (
    split,
    train,
    validation,
    train_predictors,
    validation_predictors,
    frequency_train,
    frequency_validation,
    amount_train,
    amount_validation,
    base_train,
    base_validation,
    graph_train,
    graph_validation,
    frequency_engineer,
    amount_engineer,
    graph_engineer,
)
gc.collect()

base_scaler = StandardScaler(copy=False)
base_x_train = base_scaler.fit_transform(base_x_train)
base_x_validation = base_scaler.transform(base_x_validation)
graph_scaler = StandardScaler(copy=False)
graph_x_train = graph_scaler.fit_transform(graph_x_train)
graph_x_validation = graph_scaler.transform(graph_x_validation)

Graph features generated: 8
Graph generation seconds: 60.8
Graph feature matrices: 15.3 MiB
Traced graph-generation peak: 75.0 MiB


card_device     (card_previous_unique_devices, device_previous...
card_address    (card_previous_unique_addresses, address_previ...
Name: features, dtype: object

card_device_edges       23939
card_address_edges      35243
cards_with_devices       7464
devices_with_cards       1681
cards_with_addresses    11546
addresses_with_cards      330
Name: state cardinality, dtype: int64

In [5]:
feature_sets = {
    "current_best": (),
    "plus_card_device_graph": graph_feature_groups["card_device"],
    "plus_card_address_graph": graph_feature_groups["card_address"],
    "plus_all_graph": all_graph_feature_names,
}
lightgbm_parameters = dict(
    learning_rate=0.1,
    n_estimators=400,
    num_leaves=63,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbosity=-1,
)
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment("fraud-detection-graph-features")

rows = []
for feature_set_name, graph_columns in feature_sets.items():
    if graph_columns:
        graph_indices = [all_graph_feature_names.index(column) for column in graph_columns]
        x_train = np.column_stack([base_x_train, graph_x_train[:, graph_indices]])
        x_validation = np.column_stack([base_x_validation, graph_x_validation[:, graph_indices]])
    else:
        x_train = base_x_train
        x_validation = base_x_validation

    model_started = time.perf_counter()
    with mlflow.start_run(run_name=f"graph-{feature_set_name}") as run:
        model = lgb.LGBMClassifier(**lightgbm_parameters)
        model.fit(x_train, y_train)
        probabilities = model.predict_proba(x_validation)[:, 1]
        metrics = evaluate_probabilities(y_validation, probabilities)
        model_seconds = time.perf_counter() - model_started
        mlflow.set_tags(
            {
                "stage": "graph-feature-engineering",
                "dataset": "ieee-cis",
                "split": "temporal-validation",
            }
        )
        mlflow.log_params(
            {
                "feature_set_name": feature_set_name,
                "graph_feature_count": len(graph_columns),
                "total_feature_count": x_train.shape[1],
                "learning_rate": lightgbm_parameters["learning_rate"],
                "n_estimators": lightgbm_parameters["n_estimators"],
                "num_leaves": lightgbm_parameters["num_leaves"],
            }
        )
        mlflow.log_metrics(
            {
                "pr_auc": metrics.pr_auc,
                "roc_auc": metrics.roc_auc,
                "precision_at_0_50": metrics.precision,
                "recall_at_0_50": metrics.recall,
                "f1_at_0_50": metrics.f1,
                "model_seconds": model_seconds,
            }
        )
        rows.append(
            {
                "Version": feature_set_name,
                "Graph Features": len(graph_columns),
                "Feature Count": x_train.shape[1],
                "PR-AUC": metrics.pr_auc,
                "ROC-AUC": metrics.roc_auc,
                "Precision": metrics.precision,
                "Recall": metrics.recall,
                "F1": metrics.f1,
                "Model Seconds": model_seconds,
                "MLflow Run ID": run.info.run_id,
            }
        )

    del model, probabilities
    if graph_columns:
        del x_train, x_validation
    gc.collect()

results = pd.DataFrame(rows).sort_values("PR-AUC", ascending=False).reset_index(drop=True)
display(results)

,Version,Graph Features,Feature Count,PR-AUC,ROC-AUC,Precision,Recall,F1,Model Seconds,MLflow Run ID
0,plus_all_graph,8,451,0.594611,0.923863,0.548678,0.559500,0.554036,30.561282,4379078e17c941a5886004ef1c41ff96
1,current_best,0,443,0.593938,0.919463,0.573609,0.555884,0.564608,32.133827,b025b256263c4b208fa17b7c8448aa85
2,plus_card_device_graph,4,447,0.589868,0.923236,0.550701,0.555227,0.552955,31.983977,e095e305af95453081da0e7866932088
3,plus_card_address_graph,4,447,0.582545,0.922056,0.526075,0.567061,0.545800,29.886637,ea6ab08cf4dc4cd5a70228a652543dcb


In [6]:
baseline = results.loc[results["Version"] == "current_best"].iloc[0]
graph_results = results.loc[results["Version"] != "current_best"]
best_graph = graph_results.iloc[0]
absolute_improvement = best_graph["PR-AUC"] - baseline["PR-AUC"]
relative_improvement = absolute_improvement / baseline["PR-AUC"]
helpful = graph_results.loc[graph_results["PR-AUC"] > baseline["PR-AUC"], "Version"].tolist()
unhelpful = graph_results.loc[graph_results["PR-AUC"] <= baseline["PR-AUC"], "Version"].tolist()

figure_path = PROJECT_ROOT / "reports" / "figures" / "graph_feature_ablation.png"
plot_data = results.sort_values("PR-AUC")
fig, axis = plt.subplots(figsize=(9, 5))
axis.barh(plot_data["Version"], plot_data["PR-AUC"], color="#4C78A8")
axis.axvline(baseline["PR-AUC"], color="#E45756", linestyle="--", label="Current best")
axis.set(
    title="Past-only graph feature ablation",
    xlabel="Validation PR-AUC",
    ylabel="Feature set",
)
axis.legend()
fig.tight_layout()
fig.savefig(figure_path, dpi=160)
plt.close(fig)


def markdown_table(frame):
    rows = [
        "| Version | Graph features | Total | PR-AUC | ROC-AUC | Precision | Recall | F1 |",
        "| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: |",
    ]
    for _, row in frame.iterrows():
        rows.append(
            f"| {row['Version']} | {int(row['Graph Features'])} | "
            f"{int(row['Feature Count'])} | {row['PR-AUC']:.6f} | "
            f"{row['ROC-AUC']:.6f} | {row['Precision']:.6f} | "
            f"{row['Recall']:.6f} | {row['F1']:.6f} |"
        )
    return "\n".join(rows)


feature_lines = "\n".join(
    f"- **{group}:** {', '.join(features)}" for group, features in graph_feature_groups.items()
)
run_lines = "\n".join(
    f"- **{row['Version']}:** `{row['MLflow Run ID']}`" for _, row in results.iterrows()
)
result_word = "improvement" if absolute_improvement > 0 else "degradation"
total_seconds = time.perf_counter() - notebook_started
report = f"""# Graph-Based Feature Engineering

## Entities, relationships, and features

Entities are `card1`, `DeviceInfo`, and `addr1`. Sparse state represents
`card1 <-> DeviceInfo` and `card1 <-> addr1`; no general graph framework is used.

{feature_lines}

Missing endpoints are unavailable and create neither nodes nor edges.

## Strict past-only methodology

Transactions are grouped by `TransactionDT`. Every row in one timestamp batch is scored from
state created by strictly earlier timestamps. Only after all rows are scored is the batch
committed. TRAIN initializes sparse state; VALIDATION starts strictly later and advances
sequentially without labels. No target-derived graph statistic is used.

## Observed validation results

{markdown_table(results)}

- Previous best PR-AUC: **{baseline["PR-AUC"]:.6f}**
- Best graph configuration: **{best_graph["Version"]}**
- Best graph PR-AUC: **{best_graph["PR-AUC"]:.6f}**
- Absolute PR-AUC {result_word}: **{absolute_improvement:+.6f}**
- Relative PR-AUC {result_word}: **{relative_improvement:+.2%}**
- Helpful graph sets: **{", ".join(helpful) if helpful else "none"}**
- Hurt or added little value: **{", ".join(unhelpful) if unhelpful else "none"}**

![Graph ablation](figures/graph_feature_ablation.png)

## Runtime and memory observations

- Graph feature generation: **{graph_generation_seconds:.1f} seconds**
- Graph feature matrices: **{graph_feature_memory_mb:.1f} MiB**
- Traced graph-generation peak: **{graph_peak_memory_mb:.1f} MiB**
- Experiment time before report write: **{total_seconds:.1f} seconds**
- Sparse dictionaries, sets, and counters are used; no dense adjacency matrix is built.
- An initial redundant per-ablation scaling path requested a 1.36 GiB temporary array;
  reusing the identical train-fitted base and graph matrices removed that allocation.
- Final relationship state: `{graph_state_counts}`

## MLflow runs

Experiment: `fraud-detection-graph-features`

{run_lines}

## Limitations and later hypotheses

These features capture only direct historical bipartite relationships, not multi-hop structure
or time-decayed connectivity. Any richer graph hypothesis requires a separate leakage-safe
experiment. No GNN, rules, SHAP, threshold work, or tuning is implemented here.
"""
report_path = PROJECT_ROOT / "reports" / "graph_features_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Absolute PR-AUC change: {absolute_improvement:+.6f}")
print(f"Relative PR-AUC change: {relative_improvement:+.2%}")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Figure: {figure_path.relative_to(PROJECT_ROOT)}")

Absolute PR-AUC change: +0.000673
Relative PR-AUC change: +0.11%
Report: reports\graph_features_summary.md
Figure: reports\figures\graph_feature_ablation.png


## Scope boundary

This notebook stops after graph-feature validation. It does not implement GNNs, SHAP, rules, threshold optimization, calibration, retuning, retraining, serving, or deployment.